## HIL-002 Data Inspection and Cleaning

This notebook documents a conservative cleaning workflow for the HIL-002 project boundaries snapshot. The raw ArcGIS response is preserved unchanged; derived cleaned attributes, geometry, and quality flags are exported separately.

- Source: City of Hillsboro GIS
- Dataset: Project boundaries
- Snapshot: `2026-08-26`
- Geometry: Polygon
- Coordinate system: NAD 1983 HARN StatePlane Oregon North FIPS 3601


In [8]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_VERSION = "2026-08-26"
DATA_ROOT = PROJECT_ROOT / DATA_VERSION
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
RAW_FILE = RAW_DIR / "HIL-002.json"
PROCESSED_FILE = PROCESSED_DIR / "HIL-002_cleaned.json"
MANIFEST_FILE = PROCESSED_DIR / "HIL-002_cleaning_manifest.json"

with open(RAW_FILE, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

df = pd.DataFrame([feature["attributes"] for feature in raw_data["features"]])
print(f"Loaded {RAW_FILE.name}: {len(df):,} rows, {len(df.columns):,} source fields")
print(f"Geometry type: {raw_data.get('geometryType')}")

Loaded HIL-002.json: 147 rows, 42 source fields
Geometry type: esriGeometryPolygon


## Raw Structure and Missingness

The inspection identifies completely empty fields, nullable project metadata, coded status values, identifier quality, and polygon geometry validity before deriving cleaned analysis fields.

In [9]:
import numpy as np
from shapely.geometry import Polygon


def arcgis_polygon_to_shapely(geometry):
    rings = geometry.get("rings", []) if geometry else []
    if not rings:
        return None
    try:
        return Polygon(rings[0], holes=rings[1:])
    except (TypeError, ValueError):
        return None

geometry_records = []
for feature in raw_data["features"]:
    geometry = feature.get("geometry") or {}
    rings = geometry.get("rings", [])
    polygon = arcgis_polygon_to_shapely(geometry)
    geometry_records.append({
        "geometry": polygon,
        "ring_count": len(rings),
        "vertex_count": sum(len(ring) for ring in rings),
        "source_geometry_present": bool(rings),
    })

geometry_df = pd.DataFrame(geometry_records)
geometry_df["valid_source_geometry"] = geometry_df["geometry"].map(
    lambda geometry: geometry is not None and geometry.is_valid
)
geometry_df["empty_geometry_flag"] = geometry_df["geometry"].isna()
geometry_df["multipart_or_interior_ring_flag"] = geometry_df["ring_count"] > 1
geometry_df["area"] = geometry_df["geometry"].map(
    lambda geometry: geometry.area if geometry is not None else np.nan
)
geometry_df["length"] = geometry_df["geometry"].map(
    lambda geometry: geometry.length if geometry is not None else np.nan
)

print("Geometry audit:")
display(geometry_df.drop(columns="geometry").describe(include="all"))
print("Invalid geometries:", int((~geometry_df["valid_source_geometry"] & ~geometry_df["empty_geometry_flag"]).sum()))
print("Empty geometries:", int(geometry_df["empty_geometry_flag"].sum()))

Geometry audit:


,ring_count,vertex_count,source_geometry_present,valid_source_geometry,empty_geometry_flag,multipart_or_interior_ring_flag,area,length
count,147.000000,147.000000,147,147,147,147,1.470000e+02,147.000000
unique,NaN,NaN,1,2,1,2,NaN,NaN
top,NaN,NaN,True,True,False,False,NaN,NaN
freq,NaN,NaN,147,140,147,140,NaN,NaN
mean,1.238095,88.061224,NaN,NaN,NaN,NaN,3.073054e+05,3823.890496
std,2.094786,341.087106,NaN,NaN,NaN,NaN,4.262493e+05,6075.567270
min,1.000000,5.000000,NaN,NaN,NaN,NaN,-1.063895e+06,319.908705
25%,1.000000,10.000000,NaN,NaN,NaN,NaN,5.644563e+04,1245.409305
50%,1.000000,18.000000,NaN,NaN,NaN,NaN,1.920962e+05,2636.503283
75%,1.000000,60.000000,NaN,NaN,NaN,NaN,4.155971e+05,3952.068050


Invalid geometries: 7
Empty geometries: 0


In [3]:
def arcgis_polygon_to_shapely(geometry):
    rings = geometry.get("rings", []) if geometry else []
    if not rings:
        return None
    try:
        return Polygon(rings[0], holes=rings[1:])
    except (TypeError, ValueError):
        return None

geometry_records = []
for feature in raw_data["features"]:
    geometry = feature.get("geometry") or {}
    rings = geometry.get("rings", [])
    polygon = arcgis_polygon_to_shapely(geometry)
    geometry_records.append({
        "geometry": polygon,
        "ring_count": len(rings),
        "vertex_count": sum(len(ring) for ring in rings),
        "source_geometry_present": bool(rings),
    })

geometry_df = pd.DataFrame(geometry_records)
geometry_df["valid_source_geometry"] = geometry_df["geometry"].map(
    lambda geometry: geometry is not None and geometry.is_valid
)
geometry_df["empty_geometry_flag"] = geometry_df["geometry"].isna()
geometry_df["multipart_or_interior_ring_flag"] = geometry_df["ring_count"] > 1
geometry_df["area"] = geometry_df["geometry"].map(
    lambda geometry: geometry.area if geometry is not None else np.nan
)
geometry_df["length"] = geometry_df["geometry"].map(
    lambda geometry: geometry.length if geometry is not None else np.nan
)

print("Geometry audit:")
display(geometry_df.drop(columns="geometry").describe(include="all"))
print("Invalid geometries:", int((~geometry_df["valid_source_geometry"] & ~geometry_df["empty_geometry_flag"]).sum()))
print("Empty geometries:", int(geometry_df["empty_geometry_flag"].sum()))


Geometry audit:


,ring_count,vertex_count,source_geometry_present,valid_source_geometry,empty_geometry_flag,multipart_or_interior_ring_flag,area,length
count,147.000000,147.000000,147,147,147,147,1.470000e+02,147.000000
unique,NaN,NaN,1,2,1,2,NaN,NaN
top,NaN,NaN,True,True,False,False,NaN,NaN
freq,NaN,NaN,147,140,147,140,NaN,NaN
mean,1.238095,88.061224,NaN,NaN,NaN,NaN,3.073054e+05,3823.890496
std,2.094786,341.087106,NaN,NaN,NaN,NaN,4.262493e+05,6075.567270
min,1.000000,5.000000,NaN,NaN,NaN,NaN,-1.063895e+06,319.908705
25%,1.000000,10.000000,NaN,NaN,NaN,NaN,5.644563e+04,1245.409305
50%,1.000000,18.000000,NaN,NaN,NaN,NaN,1.920962e+05,2636.503283
75%,1.000000,60.000000,NaN,NaN,NaN,NaN,4.155971e+05,3952.068050


Invalid geometries: 7
Empty geometries: 0


## Quality Findings and Derived View

No values are imputed. Blank text becomes missing in derived attributes, dates become ISO-8601 UTC strings, and quality flags identify invalid or structurally complex polygons, duplicate identifiers, and inconsistent date ordering.

In [10]:
import numpy as np
from shapely.validation import make_valid

analysis_df = df.copy()
text_columns = analysis_df.select_dtypes(include=["str"]).columns
for column in text_columns:
    analysis_df[column] = analysis_df[column].map(
        lambda value: None if isinstance(value, str) and not value.strip() else value
    )

for column in [
    "PERMIT_CREATE_DATE", "ANT_START_DATE", "ANT_COMP_DATE",
    "STATUS_CHANGE_DATE", "WARRANTY_START_DATE", "WARRANTY_END_DATE",
    "UTC_CreateDate", "UTC_EditDate",
]:
    analysis_df[column] = pd.to_datetime(
        analysis_df[column], unit="ms", utc=True, errors="coerce"
    ).dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    analysis_df[column] = analysis_df[column].where(analysis_df[column].notna(), None)

analysis_df["GEOMETRY_REPAIRED"] = ~geometry_df["valid_source_geometry"] & ~geometry_df["empty_geometry_flag"]
analysis_df["GEOMETRY_EMPTY_QA_FLAG"] = geometry_df["empty_geometry_flag"]
analysis_df["GEOMETRY_QA_FLAG"] = (
    analysis_df["GEOMETRY_REPAIRED"]
    | geometry_df["multipart_or_interior_ring_flag"]
)
analysis_df["OBJECTID_DUPLICATE_QA_FLAG"] = analysis_df["OBJECTID"].duplicated(keep=False)
analysis_df["GLOBALID_DUPLICATE_QA_FLAG"] = analysis_df["GlobalID"].duplicated(keep=False)

start_dates = pd.to_datetime(df["ANT_START_DATE"], unit="ms", utc=True, errors="coerce")
completion_dates = pd.to_datetime(df["ANT_COMP_DATE"], unit="ms", utc=True, errors="coerce")
analysis_df["DATE_ORDER_QA_FLAG"] = (
    start_dates.notna() & completion_dates.notna() & (completion_dates < start_dates)
)

completely_missing_fields = [column for column in df.columns if df[column].isna().all()]
analysis_df_compact = analysis_df.drop(columns=completely_missing_fields)
analysis_geometry_by_id = {}
for index, row in geometry_df.iterrows():
    geometry = row["geometry"]
    analysis_geometry_by_id[df.iloc[index]["OBJECTID"]] = (
        make_valid(geometry) if geometry is not None and not geometry.is_valid else geometry
    )

print("Completely missing source fields:", completely_missing_fields)
print("Derived analytical fields:", len(analysis_df_compact.columns))
print("QA flag totals:")
display(analysis_df.filter(like="_QA_FLAG").sum().to_frame("flagged_records"))

Completely missing source fields: ['WATER_INSPECTOR']
Derived analytical fields: 47
QA flag totals:


,flagged_records
GEOMETRY_EMPTY_QA_FLAG,0
GEOMETRY_QA_FLAG,7
OBJECTID_DUPLICATE_QA_FLAG,0
GLOBALID_DUPLICATE_QA_FLAG,0
DATE_ORDER_QA_FLAG,0


## Validation and Export

Rows and source geometries are preserved one-for-one. Seven invalid polygon geometries are repaired in the derived geometry view with `make_valid`; multipart or interior-ring structures are retained and flagged for contextual review.

In [11]:
assert len(raw_data["features"]) == len(analysis_df) == 147
assert analysis_df["OBJECTID"].is_unique
assert analysis_df["GlobalID"].is_unique
assert not analysis_df["GEOMETRY_EMPTY_QA_FLAG"].any()
assert all(
    geometry is not None and geometry.is_valid and geometry.area >= 0
    for geometry in analysis_geometry_by_id.values()
)
assert analysis_df_compact.columns.is_unique

print("Cleaning validation passed")
print(f"Rows preserved: {len(analysis_df):,}")
print(f"Compact fields: {len(analysis_df_compact.columns):,}")
print(f"Repaired geometries: {int(analysis_df['GEOMETRY_REPAIRED'].sum())}")
print(f"Geometry QA flags: {int(analysis_df['GEOMETRY_QA_FLAG'].sum())}")


Cleaning validation passed
Rows preserved: 147
Compact fields: 47
Repaired geometries: 7
Geometry QA flags: 7


In [12]:
import math


def json_safe(value):
    if value is None or value is pd.NA:
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and math.isnan(value):
        return None
    return value


def geometry_to_arcgis_rings(geometry):
    if geometry is None:
        return None
    polygons = [geometry] if geometry.geom_type == "Polygon" else list(geometry.geoms)
    rings = []
    for polygon in polygons:
        rings.append([[json_safe(x), json_safe(y)] for x, y in polygon.exterior.coords])
        rings.extend(
            [[[json_safe(x), json_safe(y)] for x, y in interior.coords]
             for interior in polygon.interiors]
        )
    return {"rings": rings}

schema_records = [field.copy() for field in raw_data["fields"]]
source_field_names = {field["name"] for field in schema_records}
for field_name in analysis_df_compact.columns:
    if field_name not in source_field_names:
        schema_records.append({
            "name": field_name,
            "alias": field_name,
            "type": "esriFieldTypeSmallInteger",
        })

processed_features = []
for index, raw_feature in enumerate(raw_data["features"]):
    attributes = {
        field: json_safe(value)
        for field, value in analysis_df_compact.iloc[index].to_dict().items()
    }
    object_id = df.iloc[index]["OBJECTID"]
    processed_features.append({
        "attributes": attributes,
        "geometry": geometry_to_arcgis_rings(analysis_geometry_by_id[object_id]),
    })

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
processed_data = {
    "objectIdFieldName": raw_data.get("objectIdFieldName"),
    "globalIdFieldName": raw_data.get("globalIdFieldName"),
    "geometryType": raw_data.get("geometryType"),
    "spatialReference": raw_data.get("spatialReference"),
    "fields": schema_records,
    "features": processed_features,
    "cleaning_summary": {
        "source_file": RAW_FILE.name,
        "data_version": DATA_VERSION,
        "rows": len(processed_features),
        "compact_attribute_fields": len(analysis_df_compact.columns),
        "completely_missing_source_fields": completely_missing_fields,
        "repaired_geometries": int(analysis_df["GEOMETRY_REPAIRED"].sum()),
        "geometry_qa_flags": int(analysis_df["GEOMETRY_QA_FLAG"].sum()),
    },
}

with open(PROCESSED_FILE, "w", encoding="utf-8") as file:
    json.dump(processed_data, file, indent=2, ensure_ascii=True)

In [13]:
from datetime import datetime

cleaning_manifest = [
    {
        "field_or_scope": "Blank text values",
        "action": "Represent blank or whitespace-only strings as missing in derived attributes",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "ArcGIS date fields",
        "action": "Convert epoch milliseconds to ISO-8601 UTC strings",
        "source_preserved": True,
        "review_flag": "DATE_ORDER_QA_FLAG",
    },
    {
        "field_or_scope": "WATER_INSPECTOR",
        "action": "Exclude completely missing field from compact analytical view",
        "source_preserved": True,
        "review_flag": "None",
    },
    {
        "field_or_scope": "Invalid polygon geometry",
        "action": "Use make_valid in derived geometry view",
        "source_preserved": True,
        "review_flag": "GEOMETRY_QA_FLAG",
    },
    {
        "field_or_scope": "Multipart or interior-ring geometry",
        "action": "Retain and flag for contextual review",
        "source_preserved": True,
        "review_flag": "GEOMETRY_QA_FLAG",
    },
]

manifest_data = {
    "dataset": "HIL-002",
    "dataset_name": "project_boundaries",
    "data_version": DATA_VERSION,
    "created_utc": datetime.now().astimezone().isoformat(),
    "source_file": str(RAW_FILE),
    "processed_file": str(PROCESSED_FILE),
    "rows": len(processed_features),
    "cleaning_manifest": cleaning_manifest,
}

with open(MANIFEST_FILE, "w", encoding="utf-8") as file:
    json.dump(manifest_data, file, indent=2, ensure_ascii=True)

with open(PROCESSED_FILE, "r", encoding="utf-8") as file:
    reloaded_processed_data = json.load(file)
with open(MANIFEST_FILE, "r", encoding="utf-8") as file:
    reloaded_manifest_data = json.load(file)

assert len(reloaded_processed_data["features"]) == len(raw_data["features"])
assert len(reloaded_manifest_data["cleaning_manifest"]) == len(cleaning_manifest)
assert reloaded_processed_data["features"][0]["attributes"]["OBJECTID"] == int(df.iloc[0]["OBJECTID"])
print(f"Wrote: {PROCESSED_FILE}")
print(f"Wrote: {MANIFEST_FILE}")
print("Reload validation passed")

Wrote: c:\Users\John\Documents\hillsborogis\2026-08-26\processed\HIL-002_cleaned.json
Wrote: c:\Users\John\Documents\hillsborogis\2026-08-26\processed\HIL-002_cleaning_manifest.json
Reload validation passed
